In [1]:
%run /Users/2923185/Desktop/'New NextMGT'/Codes/'T100 MGT - Steady State'/'Functions_3 - Solver.ipynb'
%run /Users/2923185/Desktop/'New NextMGT'/'Experimental Data'/'Phase 2'/Subroutines/"Experimental Data Post Process Functions.ipynb"
%run /Users/2923185/Desktop/'New NextMGT'/Codes/'T100 MGT - Steady State'/'Functions_3 - Solver.ipynb'
%run /Users/2923185/Desktop/'New NextMGT'/'Secondements'/PSI/Codes/Subroutines/'Preprocess Experimental Data'/'Experimental Data Post Process Functions_PSI Data.ipynb'
%run /Users/2923185/Desktop/'New NextMGT'/'Probe Calibration'/'Functions for Probe Calibration Project.ipynb'
%run /Users/2923185/Desktop/'New NextMGT'/'Conceptual Microgrid'/'Codes'/'Functions for Microgrid Optimization.ipynb'

%run /Users/2923185/Desktop/'New NextMGT'/'Building Heating Optimization'/'Codes'/'Functions for Building Heating Optimization.ipynb'



2023-04-24 10:59:44.147305: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
import netCDF4
from netCDF4 import Dataset
import datetime


In [3]:
# Gullfaks:

consumer_latitude = 61.21472222 
consumer_longitude = 2.27361111 

In [4]:
year_ = '2022'
month_ = '01'
day_ = '22'


# save_data_path = '/Users/2923185/Desktop/New NextMGT/Offshore Microgrid/Data/Forecast Weather Data/'+year_+month_+day_+'/'
save_data_path = '/Users/2923185/Desktop/New NextMGT/Offshore Microgrid/Data/Forecast Weather Data/'


In [5]:
filename = 'https://thredds.met.no/thredds/dodsC/metpparchive/'+year_+'/'+month_+'/'+day_+'/met_forecast_1_0km_nordic_'+year_+month_+day_+'T00Z.nc'
ncfile   = netCDF4.Dataset(filename)
file_name = filename.split('/')[-1].split('.')[0]
save_data_path_name = save_data_path + file_name + '.csv'

time_array = ncfile.variables['time'][:].data
# time_array   = ncfile.variables["time" ][:]
year = []; month = []; day = []
hour = []; minute = []; second = []

for i in range(len(time_array)):
    date_time_current = datetime.datetime.fromtimestamp(time_array[i])
    date = str(date_time_current).split(' ')[0]
    time = str(date_time_current).split(' ')[1]
    year.append(date.split('-')[0])
    month.append(date.split('-')[1])
    day.append(date.split('-')[2])
    hour.append(time.split(':')[0])
    minute.append(time.split(':')[1])
    second.append(time.split(':')[2])

year = [int(i) for i in year]; month = [int(i) for i in month]; day = [int(i) for i in day];
hour = [int(i) for i in hour]; minute = [int(i) for i in minute]; second = [int(i) for i in second];
latitude_array   = ncfile.variables['latitude'][:].data
longitude_array   = ncfile.variables['longitude'][:].data
df = pd.DataFrame()
df['latitude'] = latitude_array.flatten()
df['longitude'] = longitude_array.flatten()
distances = []
for row in range(len(df)):
    distances.append(geo_distance(df['latitude'].iloc[row],df['longitude'].iloc[row],consumer_latitude,consumer_longitude))
df['distance_from_target'] = distances
df_closest = df[df['distance_from_target'] == df['distance_from_target'].min()]
index_latitude, value_latitude = find_nearest(latitude_array, df_closest['latitude'].iloc[0])
index_longitude, value_longitude = find_nearest(longitude_array, df_closest['longitude'].iloc[0])
print(index_latitude)
print(index_longitude)
index_x = index_latitude[0]
index_y = index_latitude[1]
land_area_fraction_array = np.array([])
temperature_array = np.array([])
pressure_array = np.array([])
relative_humidity_array = np.array([])
cloud_area_fraction_array = np.array([])
wind_speed_array = np.array([])
# integral_of_surface_downwelling_longwave_flux_in_air_wrt_time_array = np.array([])
# integral_of_surface_downwelling_shortwave_flux_in_air_wrt_time_array = np.array([])
precipitation_amount_array = np.array([])
wind_direction_array = np.array([])

for time_step in range(len(time_array)):
    land_area_fraction_array = np.append(land_area_fraction_array, ncfile.variables['land_area_fraction'][index_x,index_y].data)
    temperature_array = np.append(temperature_array, ncfile.variables['air_temperature_2m'][time_step,index_x,index_y].data)
    pressure_array = np.append(pressure_array, ncfile.variables['air_pressure_at_sea_level'][time_step,index_x,index_y].data)
    relative_humidity_array = np.append(relative_humidity_array, ncfile.variables['relative_humidity_2m'][time_step,index_x,index_y].data)
    cloud_area_fraction_array = np.append(cloud_area_fraction_array, ncfile.variables['cloud_area_fraction'][time_step,index_x,index_y].data)
    wind_speed_array = np.append(wind_speed_array, ncfile.variables['wind_speed_10m'][time_step,index_x,index_y].data)
#     integral_of_surface_downwelling_longwave_flux_in_air_wrt_time_array = np.append(integral_of_surface_downwelling_longwave_flux_in_air_wrt_time_array, ncfile.variables['integral_of_surface_downwelling_longwave_flux_in_air_wrt_time'][time_step,index_x,index_y].data)
#     integral_of_surface_downwelling_shortwave_flux_in_air_wrt_time_array = np.append(integral_of_surface_downwelling_shortwave_flux_in_air_wrt_time_array, ncfile.variables['integral_of_surface_downwelling_shortwave_flux_in_air_wrt_time'][time_step,index_x,index_y].data)
    precipitation_amount_array = np.append(precipitation_amount_array, ncfile.variables['precipitation_amount'][time_step,index_x,index_y].data)
    wind_direction_array = np.append(wind_direction_array, ncfile.variables['wind_direction_10m'][time_step,index_x,index_y].data)
        
df_data = pd.DataFrame()
df_data['year'] = year; df_data['month'] = month; df_data['day'] = day;
df_data['hour'] = hour; df_data['minute'] = minute; df_data['second'] = second;
df_data['latitude'] = value_latitude*np.ones(len(year)); df_data['longitude'] = value_longitude*np.ones(len(year));
df_data['land_area_fraction'] = land_area_fraction_array
df_data['air_temperature_2m'] = temperature_array
df_data['air_pressure_at_sea_level'] = pressure_array
df_data['relative_humidity_2m'] = relative_humidity_array
df_data['cloud_area_fraction'] = cloud_area_fraction_array
df_data['wind_speed_10m'] = wind_speed_array
# df_data['integral_of_surface_downwelling_longwave_flux_in_air_wrt_time'] = integral_of_surface_downwelling_longwave_flux_in_air_wrt_time_array
# df_data['integral_of_surface_downwelling_shortwave_flux_in_air_wrt_time'] = integral_of_surface_downwelling_shortwave_flux_in_air_wrt_time_array
df_data['precipitation_amount'] = precipitation_amount_array
df_data['wind_direction_10m'] = wind_direction_array

df_data.to_csv(save_data_path_name)
del df, df_data   


(973, 220)
(973, 220)


In [6]:
filename = 'https://thredds.met.no/thredds/dodsC/metpparchive/'+year_+'/'+month_+'/'+day_+'/met_forecast_1_0km_nordic_'+year_+month_+day_+'T06Z.nc'
ncfile   = netCDF4.Dataset(filename)
file_name = filename.split('/')[-1].split('.')[0]
save_data_path_name = save_data_path + file_name + '.csv'

time_array = ncfile.variables['time'][:].data
# time_array   = ncfile.variables["time" ][:]
year = []; month = []; day = []
hour = []; minute = []; second = []

for i in range(len(time_array)):
    date_time_current = datetime.datetime.fromtimestamp(time_array[i])
    date = str(date_time_current).split(' ')[0]
    time = str(date_time_current).split(' ')[1]
    year.append(date.split('-')[0])
    month.append(date.split('-')[1])
    day.append(date.split('-')[2])
    hour.append(time.split(':')[0])
    minute.append(time.split(':')[1])
    second.append(time.split(':')[2])

year = [int(i) for i in year]; month = [int(i) for i in month]; day = [int(i) for i in day];
hour = [int(i) for i in hour]; minute = [int(i) for i in minute]; second = [int(i) for i in second];
latitude_array   = ncfile.variables['latitude'][:].data
longitude_array   = ncfile.variables['longitude'][:].data
df = pd.DataFrame()
df['latitude'] = latitude_array.flatten()
df['longitude'] = longitude_array.flatten()
distances = []
for row in range(len(df)):
    distances.append(geo_distance(df['latitude'].iloc[row],df['longitude'].iloc[row],consumer_latitude,consumer_longitude))
df['distance_from_target'] = distances
df_closest = df[df['distance_from_target'] == df['distance_from_target'].min()]
index_latitude, value_latitude = find_nearest(latitude_array, df_closest['latitude'].iloc[0])
index_longitude, value_longitude = find_nearest(longitude_array, df_closest['longitude'].iloc[0])
print(index_latitude)
print(index_longitude)
index_x = index_latitude[0]
index_y = index_latitude[1]
land_area_fraction_array = np.array([])
temperature_array = np.array([])
pressure_array = np.array([])
relative_humidity_array = np.array([])
cloud_area_fraction_array = np.array([])
wind_speed_array = np.array([])
# integral_of_surface_downwelling_longwave_flux_in_air_wrt_time_array = np.array([])
# integral_of_surface_downwelling_shortwave_flux_in_air_wrt_time_array = np.array([])
precipitation_amount_array = np.array([])
wind_direction_array = np.array([])

for time_step in range(len(time_array)):
    land_area_fraction_array = np.append(land_area_fraction_array, ncfile.variables['land_area_fraction'][index_x,index_y].data)
    temperature_array = np.append(temperature_array, ncfile.variables['air_temperature_2m'][time_step,index_x,index_y].data)
    pressure_array = np.append(pressure_array, ncfile.variables['air_pressure_at_sea_level'][time_step,index_x,index_y].data)
    relative_humidity_array = np.append(relative_humidity_array, ncfile.variables['relative_humidity_2m'][time_step,index_x,index_y].data)
    cloud_area_fraction_array = np.append(cloud_area_fraction_array, ncfile.variables['cloud_area_fraction'][time_step,index_x,index_y].data)
    wind_speed_array = np.append(wind_speed_array, ncfile.variables['wind_speed_10m'][time_step,index_x,index_y].data)
#     integral_of_surface_downwelling_longwave_flux_in_air_wrt_time_array = np.append(integral_of_surface_downwelling_longwave_flux_in_air_wrt_time_array, ncfile.variables['integral_of_surface_downwelling_longwave_flux_in_air_wrt_time'][time_step,index_x,index_y].data)
#     integral_of_surface_downwelling_shortwave_flux_in_air_wrt_time_array = np.append(integral_of_surface_downwelling_shortwave_flux_in_air_wrt_time_array, ncfile.variables['integral_of_surface_downwelling_shortwave_flux_in_air_wrt_time'][time_step,index_x,index_y].data)
    precipitation_amount_array = np.append(precipitation_amount_array, ncfile.variables['precipitation_amount'][time_step,index_x,index_y].data)
    wind_direction_array = np.append(wind_direction_array, ncfile.variables['wind_direction_10m'][time_step,index_x,index_y].data)
        
df_data = pd.DataFrame()
df_data['year'] = year; df_data['month'] = month; df_data['day'] = day;
df_data['hour'] = hour; df_data['minute'] = minute; df_data['second'] = second;
df_data['latitude'] = value_latitude*np.ones(len(year)); df_data['longitude'] = value_longitude*np.ones(len(year));
df_data['land_area_fraction'] = land_area_fraction_array
df_data['air_temperature_2m'] = temperature_array
df_data['air_pressure_at_sea_level'] = pressure_array
df_data['relative_humidity_2m'] = relative_humidity_array
df_data['cloud_area_fraction'] = cloud_area_fraction_array
df_data['wind_speed_10m'] = wind_speed_array
# df_data['integral_of_surface_downwelling_longwave_flux_in_air_wrt_time'] = integral_of_surface_downwelling_longwave_flux_in_air_wrt_time_array
# df_data['integral_of_surface_downwelling_shortwave_flux_in_air_wrt_time'] = integral_of_surface_downwelling_shortwave_flux_in_air_wrt_time_array
df_data['precipitation_amount'] = precipitation_amount_array
df_data['wind_direction_10m'] = wind_direction_array

df_data.to_csv(save_data_path_name)
del df, df_data   


(973, 220)
(973, 220)


In [7]:
filename = 'https://thredds.met.no/thredds/dodsC/metpparchive/'+year_+'/'+month_+'/'+day_+'/met_forecast_1_0km_nordic_'+year_+month_+day_+'T12Z.nc'
ncfile   = netCDF4.Dataset(filename)
file_name = filename.split('/')[-1].split('.')[0]
save_data_path_name = save_data_path + file_name + '.csv'

time_array = ncfile.variables['time'][:].data
# time_array   = ncfile.variables["time" ][:]
year = []; month = []; day = []
hour = []; minute = []; second = []

for i in range(len(time_array)):
    date_time_current = datetime.datetime.fromtimestamp(time_array[i])
    date = str(date_time_current).split(' ')[0]
    time = str(date_time_current).split(' ')[1]
    year.append(date.split('-')[0])
    month.append(date.split('-')[1])
    day.append(date.split('-')[2])
    hour.append(time.split(':')[0])
    minute.append(time.split(':')[1])
    second.append(time.split(':')[2])

year = [int(i) for i in year]; month = [int(i) for i in month]; day = [int(i) for i in day];
hour = [int(i) for i in hour]; minute = [int(i) for i in minute]; second = [int(i) for i in second];
latitude_array   = ncfile.variables['latitude'][:].data
longitude_array   = ncfile.variables['longitude'][:].data
df = pd.DataFrame()
df['latitude'] = latitude_array.flatten()
df['longitude'] = longitude_array.flatten()
distances = []
for row in range(len(df)):
    distances.append(geo_distance(df['latitude'].iloc[row],df['longitude'].iloc[row],consumer_latitude,consumer_longitude))
df['distance_from_target'] = distances
df_closest = df[df['distance_from_target'] == df['distance_from_target'].min()]
index_latitude, value_latitude = find_nearest(latitude_array, df_closest['latitude'].iloc[0])
index_longitude, value_longitude = find_nearest(longitude_array, df_closest['longitude'].iloc[0])
print(index_latitude)
print(index_longitude)
index_x = index_latitude[0]
index_y = index_latitude[1]
land_area_fraction_array = np.array([])
temperature_array = np.array([])
pressure_array = np.array([])
relative_humidity_array = np.array([])
cloud_area_fraction_array = np.array([])
wind_speed_array = np.array([])
# integral_of_surface_downwelling_longwave_flux_in_air_wrt_time_array = np.array([])
# integral_of_surface_downwelling_shortwave_flux_in_air_wrt_time_array = np.array([])
precipitation_amount_array = np.array([])
wind_direction_array = np.array([])

for time_step in range(len(time_array)):
    land_area_fraction_array = np.append(land_area_fraction_array, ncfile.variables['land_area_fraction'][index_x,index_y].data)
    temperature_array = np.append(temperature_array, ncfile.variables['air_temperature_2m'][time_step,index_x,index_y].data)
    pressure_array = np.append(pressure_array, ncfile.variables['air_pressure_at_sea_level'][time_step,index_x,index_y].data)
    relative_humidity_array = np.append(relative_humidity_array, ncfile.variables['relative_humidity_2m'][time_step,index_x,index_y].data)
    cloud_area_fraction_array = np.append(cloud_area_fraction_array, ncfile.variables['cloud_area_fraction'][time_step,index_x,index_y].data)
    wind_speed_array = np.append(wind_speed_array, ncfile.variables['wind_speed_10m'][time_step,index_x,index_y].data)
#     integral_of_surface_downwelling_longwave_flux_in_air_wrt_time_array = np.append(integral_of_surface_downwelling_longwave_flux_in_air_wrt_time_array, ncfile.variables['integral_of_surface_downwelling_longwave_flux_in_air_wrt_time'][time_step,index_x,index_y].data)
#     integral_of_surface_downwelling_shortwave_flux_in_air_wrt_time_array = np.append(integral_of_surface_downwelling_shortwave_flux_in_air_wrt_time_array, ncfile.variables['integral_of_surface_downwelling_shortwave_flux_in_air_wrt_time'][time_step,index_x,index_y].data)
    precipitation_amount_array = np.append(precipitation_amount_array, ncfile.variables['precipitation_amount'][time_step,index_x,index_y].data)
    wind_direction_array = np.append(wind_direction_array, ncfile.variables['wind_direction_10m'][time_step,index_x,index_y].data)
        
df_data = pd.DataFrame()
df_data['year'] = year; df_data['month'] = month; df_data['day'] = day;
df_data['hour'] = hour; df_data['minute'] = minute; df_data['second'] = second;
df_data['latitude'] = value_latitude*np.ones(len(year)); df_data['longitude'] = value_longitude*np.ones(len(year));
df_data['land_area_fraction'] = land_area_fraction_array
df_data['air_temperature_2m'] = temperature_array
df_data['air_pressure_at_sea_level'] = pressure_array
df_data['relative_humidity_2m'] = relative_humidity_array
df_data['cloud_area_fraction'] = cloud_area_fraction_array
df_data['wind_speed_10m'] = wind_speed_array
# df_data['integral_of_surface_downwelling_longwave_flux_in_air_wrt_time'] = integral_of_surface_downwelling_longwave_flux_in_air_wrt_time_array
# df_data['integral_of_surface_downwelling_shortwave_flux_in_air_wrt_time'] = integral_of_surface_downwelling_shortwave_flux_in_air_wrt_time_array
df_data['precipitation_amount'] = precipitation_amount_array
df_data['wind_direction_10m'] = wind_direction_array

df_data.to_csv(save_data_path_name)
del df, df_data   


(973, 220)
(973, 220)


In [8]:
filename = 'https://thredds.met.no/thredds/dodsC/metpparchive/'+year_+'/'+month_+'/'+day_+'/met_forecast_1_0km_nordic_'+year_+month_+day_+'T18Z.nc'
ncfile   = netCDF4.Dataset(filename)
file_name = filename.split('/')[-1].split('.')[0]
save_data_path_name = save_data_path + file_name + '.csv'

time_array = ncfile.variables['time'][:].data
# time_array   = ncfile.variables["time" ][:]
year = []; month = []; day = []
hour = []; minute = []; second = []

for i in range(len(time_array)):
    date_time_current = datetime.datetime.fromtimestamp(time_array[i])
    date = str(date_time_current).split(' ')[0]
    time = str(date_time_current).split(' ')[1]
    year.append(date.split('-')[0])
    month.append(date.split('-')[1])
    day.append(date.split('-')[2])
    hour.append(time.split(':')[0])
    minute.append(time.split(':')[1])
    second.append(time.split(':')[2])

year = [int(i) for i in year]; month = [int(i) for i in month]; day = [int(i) for i in day];
hour = [int(i) for i in hour]; minute = [int(i) for i in minute]; second = [int(i) for i in second];
latitude_array   = ncfile.variables['latitude'][:].data
longitude_array   = ncfile.variables['longitude'][:].data
df = pd.DataFrame()
df['latitude'] = latitude_array.flatten()
df['longitude'] = longitude_array.flatten()
distances = []
for row in range(len(df)):
    distances.append(geo_distance(df['latitude'].iloc[row],df['longitude'].iloc[row],consumer_latitude,consumer_longitude))
df['distance_from_target'] = distances
df_closest = df[df['distance_from_target'] == df['distance_from_target'].min()]
index_latitude, value_latitude = find_nearest(latitude_array, df_closest['latitude'].iloc[0])
index_longitude, value_longitude = find_nearest(longitude_array, df_closest['longitude'].iloc[0])
print(index_latitude)
print(index_longitude)
index_x = index_latitude[0]
index_y = index_latitude[1]
land_area_fraction_array = np.array([])
temperature_array = np.array([])
pressure_array = np.array([])
relative_humidity_array = np.array([])
cloud_area_fraction_array = np.array([])
wind_speed_array = np.array([])
# integral_of_surface_downwelling_longwave_flux_in_air_wrt_time_array = np.array([])
# integral_of_surface_downwelling_shortwave_flux_in_air_wrt_time_array = np.array([])
precipitation_amount_array = np.array([])
wind_direction_array = np.array([])

for time_step in range(len(time_array)):
    land_area_fraction_array = np.append(land_area_fraction_array, ncfile.variables['land_area_fraction'][index_x,index_y].data)
    temperature_array = np.append(temperature_array, ncfile.variables['air_temperature_2m'][time_step,index_x,index_y].data)
    pressure_array = np.append(pressure_array, ncfile.variables['air_pressure_at_sea_level'][time_step,index_x,index_y].data)
    relative_humidity_array = np.append(relative_humidity_array, ncfile.variables['relative_humidity_2m'][time_step,index_x,index_y].data)
    cloud_area_fraction_array = np.append(cloud_area_fraction_array, ncfile.variables['cloud_area_fraction'][time_step,index_x,index_y].data)
    wind_speed_array = np.append(wind_speed_array, ncfile.variables['wind_speed_10m'][time_step,index_x,index_y].data)
#     integral_of_surface_downwelling_longwave_flux_in_air_wrt_time_array = np.append(integral_of_surface_downwelling_longwave_flux_in_air_wrt_time_array, ncfile.variables['integral_of_surface_downwelling_longwave_flux_in_air_wrt_time'][time_step,index_x,index_y].data)
#     integral_of_surface_downwelling_shortwave_flux_in_air_wrt_time_array = np.append(integral_of_surface_downwelling_shortwave_flux_in_air_wrt_time_array, ncfile.variables['integral_of_surface_downwelling_shortwave_flux_in_air_wrt_time'][time_step,index_x,index_y].data)
    precipitation_amount_array = np.append(precipitation_amount_array, ncfile.variables['precipitation_amount'][time_step,index_x,index_y].data)
    wind_direction_array = np.append(wind_direction_array, ncfile.variables['wind_direction_10m'][time_step,index_x,index_y].data)
        
df_data = pd.DataFrame()
df_data['year'] = year; df_data['month'] = month; df_data['day'] = day;
df_data['hour'] = hour; df_data['minute'] = minute; df_data['second'] = second;
df_data['latitude'] = value_latitude*np.ones(len(year)); df_data['longitude'] = value_longitude*np.ones(len(year));
df_data['land_area_fraction'] = land_area_fraction_array
df_data['air_temperature_2m'] = temperature_array
df_data['air_pressure_at_sea_level'] = pressure_array
df_data['relative_humidity_2m'] = relative_humidity_array
df_data['cloud_area_fraction'] = cloud_area_fraction_array
df_data['wind_speed_10m'] = wind_speed_array
# df_data['integral_of_surface_downwelling_longwave_flux_in_air_wrt_time'] = integral_of_surface_downwelling_longwave_flux_in_air_wrt_time_array
# df_data['integral_of_surface_downwelling_shortwave_flux_in_air_wrt_time'] = integral_of_surface_downwelling_shortwave_flux_in_air_wrt_time_array
df_data['precipitation_amount'] = precipitation_amount_array
df_data['wind_direction_10m'] = wind_direction_array

df_data.to_csv(save_data_path_name)
del df, df_data   


(973, 220)
(973, 220)
